# 3. Microsoft Sentinel — Implementation

## Setting up Sentinel

```bash
# 1. Create Log Analytics workspace
az monitor log-analytics workspace create -g rg-security -n la-sentinel \
  --sku PerGB2018 --retention-time 90

# 2. Enable Sentinel on the workspace
az sentinel onboarding-state create -g rg-security \
  --workspace-name la-sentinel -n default
```

## Data connectors

Sentinel needs data. Connectors ingest logs from various sources:

| Connector | Data | Setup |
|-----------|------|-------|
| **Microsoft Entra ID** | Sign-in logs, audit logs | 1-click in Sentinel |
| **Microsoft 365** | Exchange, SharePoint, Teams | 1-click |
| **Defender XDR** | Incidents from all Defender products | 1-click |
| **Azure Activity** | ARM operations (who created/deleted what) | Diagnostic settings |
| **Azure Firewall** | Network traffic logs | Diagnostic settings → LA workspace |
| **CEF/Syslog** | Third-party firewalls, Linux servers | Log forwarder VM |
| **Custom logs** | Your application logs | DCR (Data Collection Rule) or API |

```bash
# Enable Azure Activity connector via diagnostic settings
az monitor diagnostic-settings create -n sentinel-activity \
  --resource /subscriptions/<sub-id> \
  --workspace la-sentinel \
  --logs '[{"category": "Administrative", "enabled": true},
          {"category": "Security", "enabled": true},
          {"category": "Alert", "enabled": true}]'
```

In [ ]:
import json
from datetime import datetime, timedelta
import random

# Simulate Sentinel analytics rules
ANALYTICS_RULES = [
    {
        'name': 'Brute force SSH',
        'severity': 'High',
        'tactic': 'CredentialAccess',
        'query': '''Syslog
| where Facility == "auth" and SyslogMessage contains "Failed password"
| summarize FailureCount=count() by HostIP, bin(TimeGenerated, 1h)
| where FailureCount > 10''',
        'frequency': 'PT1H',
        'lookback': 'PT1H',
        'trigger_threshold': 0,
    },
    {
        'name': 'Suspicious Azure AD sign-in',
        'severity': 'Medium',
        'tactic': 'InitialAccess',
        'query': '''SigninLogs
| where ResultType != 0
| where IPAddress !startswith "10."
| summarize FailedAttempts=count() by UserPrincipalName, IPAddress
| where FailedAttempts > 5''',
        'frequency': 'PT5M',
        'lookback': 'PT1H',
        'trigger_threshold': 0,
    },
    {
        'name': 'Mass file deletion in SharePoint',
        'severity': 'High',
        'tactic': 'Impact',
        'query': '''OfficeActivity
| where Operation == "FileDeleted"
| summarize DeleteCount=count() by UserId, bin(TimeGenerated, 1h)
| where DeleteCount > 100''',
        'frequency': 'PT1H',
        'lookback': 'PT1H',
        'trigger_threshold': 0,
    },
    {
        'name': 'Key Vault secret accessed from unusual IP',
        'severity': 'Medium',
        'tactic': 'CredentialAccess',
        'query': '''AzureDiagnostics
| where ResourceType == "VAULTS" and OperationName == "SecretGet"
| where CallerIPAddress !startswith "10."
| project TimeGenerated, Resource, CallerIPAddress, identity_claim_upn_s''',
        'frequency': 'PT15M',
        'lookback': 'PT1H',
        'trigger_threshold': 0,
    },
]

print('=== Sentinel Analytics Rules ===\n')
for rule in ANALYTICS_RULES:
    sev_icon = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}[rule['severity']]
    print(f'{sev_icon} {rule["name"]}')
    print(f'   Severity: {rule["severity"]}  |  Tactic: {rule["tactic"]}')
    print(f'   Runs every: {rule["frequency"]}  |  Looks back: {rule["lookback"]}')
    print(f'   KQL:')
    for line in rule['query'].split('\n'):
        print(f'     {line}')
    print()

## Creating analytics rules via CLI

```bash
# Create a scheduled analytics rule
az sentinel alert-rule create -g rg-security \
  --workspace-name la-sentinel --rule-name brute-force-ssh \
  --type Scheduled \
  --display-name "Brute force SSH" \
  --description "Detects >10 failed SSH attempts per hour" \
  --severity High \
  --query 'Syslog | where Facility == "auth" and SyslogMessage contains "Failed password" | summarize count() by HostIP, bin(TimeGenerated, 1h) | where count_ > 10' \
  --query-frequency PT1H \
  --query-period PT1H \
  --trigger-operator GreaterThan \
  --trigger-threshold 0 \
  --tactics CredentialAccess \
  --enabled true
```

## Automation with playbooks

When an analytics rule fires, trigger a **playbook** (Logic App):

```bash
# Create automation rule that triggers a playbook
az sentinel automation-rule create -g rg-security \
  --workspace-name la-sentinel --automation-rule-name auto-block-ip \
  --order 1 \
  --triggering-logic '{"isEnabled": true, "triggersOn": "Incidents", "triggersWhen": "Created", "conditions": [{"conditionType": "Property", "conditionProperties": {"propertyName": "IncidentSeverity", "operator": "Equals", "propertyValues": ["High"]}}]}' \
  --actions '[{"actionType": "RunPlaybook", "actionConfiguration": {"logicAppResourceId": "/subscriptions/.../playbook-block-ip"}}]'
```

In [ ]:
# Simulate Sentinel incident workflow
PLAYBOOK_ACTIONS = {
    'brute-force': [
        'Block source IP in Azure Firewall',
        'Disable compromised user account',
        'Send alert to SOC Teams channel',
        'Create ServiceNow incident P2',
    ],
    'data-exfil': [
        'Revoke user sessions',
        'Block user from SharePoint',
        'Notify manager via email',
        'Escalate to insider risk investigation',
    ],
}

def simulate_incident(rule_name: str, entities: dict) -> dict:
    now = datetime.now()
    incident = {
        'id': f'INC-{random.randint(1000,9999)}',
        'title': rule_name,
        'created': now.strftime('%Y-%m-%d %H:%M'),
        'status': 'New',
        'entities': entities,
    }
    
    playbook = PLAYBOOK_ACTIONS.get('brute-force' if 'brute' in rule_name.lower() else 'data-exfil', ['Manual investigation required'])
    incident['automated_response'] = playbook
    return incident

print('=== Sentinel Incident Response Flow ===\n')
print('1. Analytics rule fires → Incident created')
print('2. Automation rule evaluates → Playbook triggered')
print('3. Playbook executes response actions\n')

incident = simulate_incident('Brute force SSH', {'source_ip': '185.220.101.42', 'target_host': 'vm-web-01', 'attempts': 47})
print(json.dumps(incident, indent=2))

print('\n--- Playbook execution ---')
for i, action in enumerate(incident['automated_response'], 1):
    print(f'  Step {i}: ✅ {action}')

## Workflow automation in Defender for Cloud

Separate from Sentinel playbooks — Defender for Cloud has its own automation for security alerts:

```bash
# Create workflow automation for high-severity Defender alerts
az security automation create -g rg-security -n auto-high-alerts \
  --scopes '[{"scopePath": "/subscriptions/<sub-id>"}]' \
  --sources '[{"eventSource": "Assessments", "ruleSets": [{"rules": [{"propertyJPath": "properties.status.code", "propertyType": "String", "expectedValue": "Unhealthy", "operator": "Equals"}]}]}]' \
  --actions '[{"logicAppResourceId": "/subscriptions/.../logicApps/notify-security-team", "actionType": "LogicApp"}]'
```

## Data collection rules (DCRs)

DCRs define what data to collect from Azure VMs and where to send it:

```bash
# Create data collection rule for security events
az monitor data-collection rule create -g rg-security -n dcr-security \
  --location eastus \
  --data-flows '[{"destinations": ["la-sentinel"], "streams": ["Microsoft-SecurityEvent"]}]' \
  --destinations '{"logAnalytics": [{"name": "la-sentinel", "workspaceResourceId": "/subscriptions/.../workspaces/la-sentinel"}]}' \
  --data-sources '{"windowsEventLogs": [{"name": "security-events", "streams": ["Microsoft-SecurityEvent"], "xPathQueries": ["Security!*[System[(EventID=4624 or EventID=4625 or EventID=4688)]]"]}]}'
```

---
## AZ-500 Domain 4 Summary

| Implementation | Key details |
|---------------|-------------|
| **Azure Policy** | Define → Initiative → Assign → Remediate. Effects: audit/deny/modify/deployIfNotExists. |
| **Key Vault** | RBAC auth, soft delete + purge protection, PE, rotation policies, backup (same tenant/geo). |
| **Defender plans** | Enable per workload. Servers P1 vs P2. Multi-cloud via security connectors. |
| **Secure Score** | Weighted by severity. Prioritize high findings. |
| **EASM** | External attack surface discovery. Seed with domains. |
| **Sentinel setup** | Log Analytics workspace + onboarding. Data connectors for each source. |
| **Analytics rules** | Scheduled KQL queries. Frequency + lookback + threshold. |
| **Playbooks** | Logic Apps triggered by automation rules. |
| **DCRs** | Define data collection from VMs to Log Analytics. |

---
## You've completed all AZ-500 labs!

### Next steps

1. Take the [AZ-500 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/azure-security-engineer/practice/assessment?assessment-type=practice&assessmentId=57&practice-assessment-type=certification)
2. Deploy the Azure CLI examples in a free Azure subscription
3. Focus on the scenarios — the exam is about *choosing the right tool for the job*
4. Review the [Microsoft Cloud Security Benchmark](https://learn.microsoft.com/en-us/security/benchmark/azure/overview)